# 6-3. 인컨텍스트 러닝 — 가중치를 고치지 않고 배우기

> **이 노트북에서 답할 질문**
> "예시 몇 개만 보여 줬는데 모델이 새 일을 해냈다"는 게 대체 어떻게 가능한가?

## 1. 무엇이 놀라운 일인가

GPT-3 논문(2020)이 충격을 준 건 성능 수치가 아니라 이것이었습니다.

```
한국 -> 서울
일본 -> 도쿄
프랑스 -> ?
```

모델은 "수도를 답하라"는 **훈련을 받은 적이 없습니다.**
그런데 예시 두 개를 보고 규칙을 알아채고 `파리`라고 답합니다.

**그리고 이 과정에서 모델의 가중치는 단 하나도 바뀌지 않습니다.**
학습이라고 부르는데 경사 하강법이 없습니다. 이게 이상한 지점입니다.

| | 파인튜닝 | 인컨텍스트 러닝 |
|---|---|---|
| 가중치 | **바뀜** | 안 바뀜 |
| 필요한 것 | GPU, 데이터셋, 시간 | 프롬프트 한 줄 |
| 지속성 | 영구적 | 그 대화 안에서만 |
| 비용 | 수만~수백만 원 | 토큰 값 |

## 2. 어떻게 가능한가 — 사전 훈련의 부산물

핵심은 이것입니다.

> 사전 훈련은 "다음 토큰 맞히기"만 시킵니다.
> 그런데 **인터넷 텍스트에는 "패턴이 반복되는 글"이 아주 많습니다.**
> 목록, 표, 번역 대조, 질문-답변 모음, 코드의 반복 구조…
>
> 그런 글에서 다음 토큰을 잘 맞히려면, 모델은 **"앞에 나온 패턴을 찾아서 이어 쓰는 능력"**
> 자체를 획득해야 합니다.

즉 인컨텍스트 러닝은 따로 가르친 기능이 아니라 **다음 토큰 예측을 잘하려다 보니 생긴 능력**입니다.

### 이걸 직접 만들어 봅시다

말로만 하면 안 믿기니, **아주 작은 트랜스포머에게 이 능력이 생기는 걸 직접 관찰**하겠습니다.

**과제**: 무작위 기호 나열이 **일정 주기로 반복**됩니다.

```
주기 5 :  D K A P M  D K A P M  D K A P M
주기 3 :  Q R T  Q R T  Q R T  Q R T ...
                 ↑ 한 바퀴 돈 뒤부터는 앞을 "복사"하면 맞음
```

두 가지 장치를 일부러 넣었습니다.

1. **기호가 매번 무작위** → 내용을 외울 수 없습니다
2. **주기 P가 샘플마다 다름** → "16칸 앞을 보라" 같은 **위치 규칙으로도 못 풉니다**

2번이 핵심입니다. 모델은 **문맥을 읽어서 "지금 몇 칸 앞을 봐야 하는지" 스스로 알아내야** 합니다.
이것 없이는 위치 임베딩만으로 편법을 쓸 수 있거든요.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from common import set_seed, get_device, count_params, human

set_seed(1234)
device = get_device()

In [ ]:
VOCAB  = 32          # 기호 종류
SEQLEN = 48          # 전체 길이
PMIN, PMAX = 4, 16   # 반복 주기(샘플마다 무작위)

def make_batch(batch_size, gen=None):
    """주기 P 로 반복되는 나열. P 는 샘플마다 다릅니다."""
    P = torch.randint(PMIN, PMAX + 1, (batch_size,), generator=gen)
    base = torch.randint(0, VOCAB, (batch_size, PMAX), generator=gen)
    idx = torch.arange(SEQLEN).unsqueeze(0) % P.unsqueeze(1)    # 0,1,..,P-1,0,1,..
    seq = base.gather(1, idx)
    return seq[:, :-1].to(device), seq[:, 1:].to(device), P.to(device)

g = torch.Generator().manual_seed(0)
x, y, P = make_batch(2, g)
for i in range(2):
    print(f"주기 P={P[i].item():>2} | {x[i].tolist()}")
print()
print(f"각 샘플의 주기가 다릅니다 -> '몇 칸 앞을 보라'를 위치로 고정할 수 없습니다.")
print(f"처음 P개는 무작위라 예측 불가 (기준선 = 1/{VOCAB} = {1/VOCAB:.1%}).")
print(f"그 뒤로는 'P칸 앞'을 복사하면 맞지만, P를 문맥에서 알아내야 합니다.")

## 3. 모델 — 2층짜리 아주 작은 트랜스포머

**층이 2개 이상이어야 합니다.** 뒤에서 이유가 드러납니다.

In [ ]:
class Attn(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.last_attn = None         # 시각화를 위해 저장해 둡니다

    def forward(self, x, save=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * hd ** -0.5
        mask = torch.tril(torch.ones(T, T, device=x.device))
        att = att.masked_fill(mask == 0, float("-inf")).softmax(-1)
        if save:
            self.last_attn = att.detach()
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class TinyTransformer(nn.Module):
    def __init__(self, vocab=VOCAB, n_embd=64, n_head=4, n_layer=2, block=SEQLEN):
        super().__init__()
        self.tok = nn.Embedding(vocab, n_embd)
        self.pos = nn.Embedding(block, n_embd)
        self.attns = nn.ModuleList([Attn(n_embd, n_head) for _ in range(n_layer)])
        self.ffns = nn.ModuleList([
            nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
            for _ in range(n_layer)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layer)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab, bias=False)

    def forward(self, idx, targets=None, save=False):
        B, T = idx.shape
        h = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for a, f, l1, l2 in zip(self.attns, self.ffns, self.ln1, self.ln2):
            h = h + a(l1(h), save=save)
            h = h + f(l2(h))
        logits = self.head(self.ln_f(h))
        if targets is None:
            return logits, None
        return logits, F.cross_entropy(logits.reshape(-1, VOCAB), targets.reshape(-1))

set_seed(1234)
model = TinyTransformer().to(device)
print(f"파라미터: {human(count_params(model))}")

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

@torch.no_grad()
def accuracy(model, n=200):
    """'처음 보는 구간'과 '반복 구간'의 정확도를 따로 잽니다."""
    model.eval()
    xb, yb, P = make_batch(n)
    correct = (model(xb)[0].argmax(-1) == yb)
    # 정답 토큰의 위치가 P 이상이면 = 이미 한 번 나온 적 있는 토큰 = 복사 가능
    tgt_pos = torch.arange(1, SEQLEN, device=device).unsqueeze(0)
    seen = tgt_pos >= P.unsqueeze(1)
    model.train()
    return correct[~seen].float().mean().item(), correct[seen].float().mean().item()

print(f"{'step':>6} | {'loss':>7} | {'처음 보는 구간':>14} | {'반복 구간':>12}")
print("-" * 50)
for step in range(3001):
    xb, yb, _ = make_batch(64)
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if step % 500 == 0:
        f, sec = accuracy(model)
        print(f"{step:>6} | {loss.item():>7.4f} | {f:>13.1%} | {sec:>11.1%}")

## 4. 결과가 말하는 것

두 숫자를 나란히 보세요.

- **처음 보는 구간**: 무작위 기호라 예측이 원리적으로 불가능합니다. $1/32 \approx 3\%$ 근처에 머뭅니다.
- **반복 구간**: 95% 이상으로 올라갑니다.

**이게 인컨텍스트 러닝입니다.**

모델은 특정 기호 나열을 외운 게 아닙니다. 매 배치마다 완전히 새로운 무작위 나열이 나오니까요.
대신 **"앞에서 본 걸 찾아서 이어 쓴다"는 알고리즘**을 가중치 안에 심었습니다.

이제 **한 번도 본 적 없는 나열**로 확인해 봅시다.

In [ ]:
# 시각화를 위해 주기를 7로 고정한 샘플을 하나 만듭니다
set_seed(999)
PERIOD = 7
base = torch.randint(0, VOCAB, (1, PERIOD))
seq = base[:, torch.arange(SEQLEN) % PERIOD]
xb, yb = seq[:, :-1].to(device), seq[:, 1:].to(device)

logits, _ = model(xb, save=True)
pred = logits.argmax(-1)

print(f"주기 P = {PERIOD}")
print("입력      :", xb[0].tolist())
print("모델 예측 :", pred[0].tolist())
print()
mark = "".join("O" if p == t else "." for p, t in zip(pred[0].tolist(), yb[0].tolist()))
print("맞음(O)/틀림(.):")
print(f"  {mark[:PERIOD]}  <- 첫 한 바퀴 (처음 보는 기호, 맞힐 수 없음)")
print(f"  {mark[PERIOD:]}  <- 그 이후 (P칸 앞을 복사하면 됨)")

## 5. 어떻게 하는지 들여다보기 — 인덕션 헤드

모델 내부에서 무슨 일이 일어나는 걸까요? 어텐션 맵을 보면 답이 나옵니다.

이 알고리즘을 **어텐션으로** 구현하려면 두 단계가 필요합니다.

**1층 — "이전 토큰 헤드"**: 각 위치가 바로 앞 토큰이 무엇이었는지 기억합니다.
그래야 각 토큰이 "나는 X 다음에 오는 토큰"이라는 꼬리표를 갖게 됩니다.

**2층 — "인덕션 헤드"**: 현재 토큰이 A라면,
**앞에서 A가 나왔던 자리의 바로 다음 토큰**을 찾아가 복사합니다.

```
  D K A P M D K A ?
      ↑       └─┘  현재 토큰은 A
      │
      └── 앞에서 A가 나왔던 자리를 "내용으로" 찾고
        ↓
        그 바로 다음 토큰 P 를 복사 -> 정답!
```

**주목**: 이 절차 어디에도 "몇 칸 앞"이라는 숫자가 없습니다.
내용을 보고 찾기 때문에 주기 P가 얼마든 작동합니다. 그래서 우리 과제를 풀 수 있는 것입니다.

이것이 **2층 이상이 필요한 이유**입니다. 1층으로는 이 두 단계를 못 합니다.
실제 LLM에서도 발견되는 구조이고, `induction head` 라는 이름이 붙어 있습니다.

눈으로 확인해 봅시다.

In [ ]:
import matplotlib.pyplot as plt

att0 = model.attns[0].last_attn[0]    # 1층 (n_head, T, T)
att1 = model.attns[1].last_attn[0]    # 2층

# 각 층에서 가장 뚜렷한 패턴을 보이는 헤드를 고릅니다
T = att0.shape[-1]
prev_score = [att0[h].diagonal(offset=-1).mean().item() for h in range(att0.shape[0])]
best0 = max(range(len(prev_score)), key=lambda h: prev_score[h])

# 인덕션: 위치 t(>=HALF)가 t-HALF+1 을 보는지
ind_score = []
for h in range(att1.shape[0]):
    # 위치 t 가 "t - PERIOD + 1"(= 같은 기호가 지난번 나온 자리의 다음)을 보는가
    vals = [att1[h, t, t - PERIOD + 1].item() for t in range(PERIOD, T)]
    ind_score.append(sum(vals) / len(vals))
best1 = max(range(len(ind_score)), key=lambda h: ind_score[h])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, m, title in zip(
        axes,
        [att0[best0].cpu(), att1[best1].cpu()],
        [f"1층 헤드 {best0}: 이전 토큰 주목 (강도 {prev_score[best0]:.2f})",
         f"2층 헤드 {best1}: 인덕션 (강도 {ind_score[best1]:.2f})"]):
    im = ax.imshow(m, cmap="viridis")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("보는 대상 (key)"); ax.set_ylabel("보는 주체 (query)")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

print(f"1층 각 헤드의 '바로 앞 토큰' 주목도 : {[round(v,2) for v in prev_score]}")
print(f"2층 각 헤드의 '인덕션' 주목도       : {[round(v,2) for v in ind_score]}")

### 그림 읽는 법

- **왼쪽(1층)**: 주대각선 바로 아래가 밝으면 "각 위치가 직전 토큰을 보고 있다"는 뜻입니다.
- **오른쪽(2층)**: 주대각선과 **평행한 밝은 선**이 `P-1` 칸 떨어져 나타납니다.
  "P칸 앞의 다음 자리를 보고 있다" — 정확히 인덕션 동작입니다.

숫자로 보면 1층의 '직전 토큰' 주목도가 0.4~0.6까지 올라가 있을 겁니다. 아주 뚜렷하죠.
2층의 인덕션 주목도는 0.1~0.2 정도로 더 낮게 나오는데, 이것도 충분히 큰 값입니다 —
40여 개 위치에 골고루 나눠 본다면 자리당 0.02 남짓일 테니까요.
그리고 이 일을 **헤드 하나가 전담하지 않고 여러 헤드가 나눠 맡기도** 합니다.

> 헤드마다 역할이 갈리는 게 보이시나요? 5-2에서 "헤드를 여러 개 두는 이유"라고 했던 것이
> 실제로 이렇게 나타납니다. 어떤 헤드는 놀고, 어떤 헤드는 특정 역할을 맡습니다.

## 6. 규모와 능력 — 창발

이 작은 실험을 실제 LLM으로 확장하면 이렇게 됩니다.

인덕션 헤드는 **학습 중 어느 시점에 갑자기 형성**되고, 그 순간 손실 곡선에 작은 턱이 생깁니다.
그리고 그때부터 모델이 인컨텍스트 러닝을 하기 시작합니다.

이것이 **창발(emergence)** 이라 불리는 현상의 한 사례입니다.
능력이 서서히 좋아지는 게 아니라 **어느 규모/시점에서 갑자기 나타납니다.**

우리 실험에서도 확인할 수 있습니다.

In [ ]:
# 1층 모델은 이 과제를 풀 수 있을까?
print("층 수를 바꿔 가며 같은 과제를 학습시켜 봅니다.\n")
print(f"{'층 수':>6} | {'파라미터':>9} | {'반복 구간 정확도':>16}")
print("-" * 38)
for n_layer in (1, 2, 3):
    set_seed(1234)
    m = TinyTransformer(n_layer=n_layer).to(device)
    o = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=0.01)
    for _ in range(3000):
        xb, yb, _ = make_batch(64)
        _, l = m(xb, yb)
        o.zero_grad(set_to_none=True); l.backward(); o.step()
    _, sec = accuracy(m)
    print(f"{n_layer:>6} | {human(count_params(m)):>9} | {sec:>15.1%}")

**1층 모델은 확실히 뒤처지고, 2층부터 급격히 올라갑니다.**

파라미터를 조금 더 준다고 되는 게 아닙니다. 1층은 "직전 토큰 기억"과 "과거 위치 찾아가기"를
**동시에** 할 수 없어서, 구조적으로 막혀 있습니다.

"조금 더 크게 만들었더니 갑자기 되더라"의 아주 작은 버전을 직접 보신 겁니다.

## 7. 실무로 연결하기

지금까지 본 원리가 프롬프트 작성법으로 직결됩니다.

### 왜 예시(few-shot)가 효과적인가
모델은 **앞에 나온 패턴을 찾아 이어 씁니다.** 예시는 그 패턴을 명시적으로 깔아 주는 것입니다.

```
좋음:                          나쁨:
입력: 오늘 날씨 좋다           "문장의 감정을 분류해줘"
출력: 긍정                      (형식이 정해지지 않음
                                 -> 모델이 형식을 추측해야 함)
입력: 최악이었다
출력: 부정

입력: 그럭저럭이네
출력:
```

### 실무 지침

| 지침 | 이 노트북의 근거 |
|---|---|
| 출력 형식을 예시로 보여라 | 모델은 패턴을 복사한다 |
| 예시의 형식을 일관되게 하라 | 패턴이 흐트러지면 복사할 게 없다 |
| 중요한 내용은 프롬프트 끝에 두라 | 가까운 토큰에 어텐션이 더 간다 |
| 예시가 많다고 항상 좋진 않다 | 문맥이 길어지면 $O(T^2)$ 비용 + 주의 분산 |

### 인컨텍스트 러닝의 한계 — 그리고 파인튜닝이 필요한 이유

| 한계 | 설명 |
|---|---|
| **휘발성** | 대화가 끝나면 사라짐. 매번 프롬프트에 다시 넣어야 함 |
| **비용** | 예시가 매 요청마다 토큰으로 과금됨 |
| **문맥 한계** | 예시 1,000개는 넣을 수 없음 |
| **깊이 한계** | 말투·문체를 일관되게 바꾸는 건 예시 몇 개로 안 됨 |

**이 네 가지가 곧 파인튜닝을 하는 이유입니다.**
프롬프트에 매번 넣던 것을 **가중치 안으로 옮기는 작업**이 파인튜닝입니다.

---

## 정리

- 인컨텍스트 러닝은 **가중치를 고치지 않고** 프롬프트만으로 새 일을 시키는 것
- 원리는 **패턴 복사**. 사전 훈련이 "다음 토큰 맞히기"를 시키다 보니 생긴 부산물
- 구현체는 **인덕션 헤드**: 1층이 직전 토큰을 기억하고, 2층이 과거 등장 위치의 다음을 복사
- **2층 이상이 필요**하고, 학습 중 어느 순간 갑자기 형성됨 (창발)
- 한계(휘발성·비용·문맥·깊이)가 곧 **파인튜닝의 존재 이유**

## 연습

1. `PMAX` 를 32로 늘려 보세요. 긴 주기일수록 학습이 오래 걸리나요?
2. `make_batch` 에서 주기를 고정(P를 항상 16)하고 1층 모델을 학습시켜 보세요.
   **1층도 100%가 나옵니다** — 위치만으로 풀 수 있게 되어 인덕션이 필요 없어지기 때문입니다.
   과제 설계가 결론을 바꾼다는 걸 직접 확인하는 연습입니다.
3. 반복 구간에 **노이즈를 섞어** 보세요(몇 개 기호를 다른 걸로 교체).
   모델이 여전히 복사를 시도하나요? LLM의 "프롬프트에 끌려가는" 현상과 닮았습니다.

## 다음

여기까지가 **사전 훈련**의 세계입니다.
다음 단계는 이 모델을 **대화할 수 있는 조수**로 바꾸는 일 — 파인튜닝(SFT)입니다.
`../../README.md` 의 로드맵을 참고하세요.